# LangChain Agents, end to end

One notebook. Eight ideas, each one building on the last:

| # | Idea | The question it answers |
|---|---|---|
| 1 | Models | How do I get an LLM object? |
| 2 | Tools | How does the model reach my code? |
| 3 | `create_agent` + system prompt | How do I get a loop that decides for itself? |
| 4 | Memory | How does it remember the last turn? |
| 5 | `invoke` / `batch` / `stream` | How do I call it? |
| 6 | `AgentState` | Where does non-message data live? |
| 7 | Structured output | How do I get back something my code can use? |
| 8 | Use case | A content writer agent using all of it |

Versions this was written and run against: `langchain 1.3`, `langgraph 1.2`.

In [ ]:
# Run once, then restart the kernel.
!pip install -q -U "langchain>=1.3" "langgraph>=1.2" "langchain-openai>=1.1" python-dotenv

---
## 1. The model

Two ways to get a chat model. You will see both in the wild.

In [2]:
from dotenv import load_dotenv, find_dotenv
load_dotenv(find_dotenv(usecwd=True))          # reads OPENAI_API_KEY from .env

# Way 1 - import the provider class directly.
from langchain_openai import ChatOpenAI
llm_direct = ChatOpenAI(model="gpt-4.1-mini", temperature=0)

# Way 2 - init_chat_model: one string, "provider:model".
from langchain.chat_models import init_chat_model
llm = init_chat_model("openai:gpt-4.1-mini", temperature=0)

print(type(llm_direct).__name__, "|", type(llm).__name__)
print(llm.invoke("Reply with one word: ready?").content)

ChatOpenAI | ChatOpenAI


Ready.


Both give you the **same object**. The difference is where the provider name lives:

- `ChatOpenAI(...)` hard-codes the provider in an `import`. Swapping to Claude means editing imports.
- `init_chat_model("openai:gpt-4.1-mini")` puts the provider in a **string** — so it can come from
  config, from `.env`, or from a dropdown in your app. Swapping to
  `"anthropic:claude-sonnet-4-6"` is a one-character-per-word change and nothing else moves.

Use `init_chat_model` unless you need a provider-only argument that the generic path doesn't expose.
We use `llm` for the rest of this notebook.

---
## 2. Tools

A tool is a Python function the model is allowed to ask for. You do not call it — the model
requests it by name, LangChain runs it, and the result goes back into the conversation.

### 2a. `@tool` — the decorator

In [3]:
from langchain.tools import tool


@tool
def word_count(text: str) -> int:
    """Count the number of words in a piece of text."""
    return len(text.split())


print("name        :", word_count.name)
print("description :", word_count.description)
print("args        :", word_count.args)

name        : word_count
description : Count the number of words in a piece of text.


args        : {'text': {'title': 'Text', 'type': 'string'}}


Three things were derived automatically, and each one matters:

| Source in your code | Becomes |
|---|---|
| function name | tool name the model calls |
| **docstring** | tool description — this is the *only* thing telling the model when to use it |
| **type hints** | the input schema (`text: str`) |

Type hints are not optional. Without them there is no schema and the tool is unusable.
The docstring is not decoration either — a vague docstring is the single most common cause of
"my agent never calls my tool".

### 2b. `tool()` — the function

`@tool` only works when you are *writing* the function. `tool()` is the same thing called
directly, which is what you need when the function already exists — someone else's code, a
`lambda`, or a function you can't add a docstring to.

In [4]:
# Pretend this came from an existing utils.py: no docstring, no LangChain in sight.
def slugify(title: str) -> str:
    return title.lower().strip().replace(" ", "-")


slug_tool = tool(
    "slugify",                                            # name, overrides the function name
    description="Turn a blog title into a URL slug.",     # description, since there is no docstring
)(slugify)

print("name        :", slug_tool.name)
print("description :", slug_tool.description)
print("result      :", slug_tool.invoke({"title": "Hello Big World"}))

name        : slugify
description : Turn a blog title into a URL slug.
result      : hello-big-world


`@tool` and `tool()` are literally the same function — a decorator *is* a function call.
`@tool` above the `def` is just Python sugar for `word_count = tool(word_count)`.

- **`@tool`** — you own the function. Default choice.
- **`tool(name, description=...)(fn)`** — the function is not yours, or you want a name and
  description different from the code's.

`tool.invoke({...})` takes a **dict** matching the schema, not positional arguments.

### 2c. What the model actually receives

Worth looking at once, so tools stop feeling like magic. This is the JSON that goes over the wire:

In [5]:
import json
from langchain_core.utils.function_calling import convert_to_openai_tool

print(json.dumps(convert_to_openai_tool(word_count), indent=2))

{
  "type": "function",
  "function": {
    "name": "word_count",
    "description": "Count the number of words in a piece of text.",
    "parameters": {
      "properties": {
        "text": {
          "type": "string"
        }
      },
      "required": [
        "text"
      ],
      "type": "object"
    }
  }
}


---
## 3. The agent, and the system prompt

`create_agent` gives you the loop: call model → if it asked for tools, run them and feed the
results back → repeat → stop when the model answers without asking for a tool.

In [6]:
from langchain.agents import create_agent

WRITER_SYSTEM = (
    "You are an editorial assistant. "
    "When a question is about the length of a text, use the word_count tool - never estimate. "
    "Answer in one short sentence."
)

agent = create_agent(
    model=llm,
    tools=[word_count, slug_tool],
    system_prompt=WRITER_SYSTEM,
)

result = agent.invoke({"messages": [{"role": "user", "content":
    "How many words is 'the quick brown fox jumps over the lazy dog'?"}]})

print(result["messages"][-1].content)

The phrase "the quick brown fox jumps over the lazy dog" contains 9 words.


The input is a **dict** with a `messages` key, and the output is a dict too. The final answer is
`result["messages"][-1].content`. Everything in between is the trace:

In [7]:
for m in result["messages"]:
    m.pretty_print()

================================ Human Message =================================

How many words is 'the quick brown fox jumps over the lazy dog'?
================================== Ai Message ==================================
Tool Calls:
  word_count (call_iZygVzEbj3OLJQ1UQMndsDdu)
 Call ID: call_iZygVzEbj3OLJQ1UQMndsDdu
  Args:
    text: the quick brown fox jumps over the lazy dog
================================= Tool Message =================================
Name: word_count

9
================================== Ai Message ==================================

The phrase "the quick brown fox jumps over the lazy dog" contains 9 words.


Four messages: your question, the model asking for `word_count`, the tool's answer, the model's
reply. That is the whole agent loop, visible.

**Notice what is missing: there is no system message in that list.**

In [8]:
print([type(m).__name__ for m in result["messages"]])

['HumanMessage', 'AIMessage', 'ToolMessage', 'AIMessage']


`system_prompt` is not a message in the conversation — it is prepended fresh on **every** model
call inside the loop. Consequences you should know:

- It does not accumulate, and it never gets summarised away.
- Changing it changes every future turn, including in threads that already exist.
- It is the right place for the agent's *job*, and the wrong place for facts about the user
  (those belong in state or memory, sections 4 and 6).

---
## 4. Memory

Look at what the agent above does across two calls.

In [9]:
agent.invoke({"messages": [{"role": "user", "content": "My name is Shiva."}]})
second = agent.invoke({"messages": [{"role": "user", "content": "What is my name?"}]})
print(second["messages"][-1].content)

I don't have information about your name. Could you please tell me?


Gone. Each `invoke` starts from an empty message list — the agent is a pure function of its input.
That is not a bug; it is what makes agents easy to scale. Memory is opt-in.

You add it with two pieces:

1. a **checkpointer** — where the conversation is written
2. a **`thread_id`** — which conversation to write to

In [10]:
from langgraph.checkpoint.memory import InMemorySaver

mem_agent = create_agent(
    model=llm,
    tools=[word_count, slug_tool],
    system_prompt=WRITER_SYSTEM,
    checkpointer=InMemorySaver(),       # 1. where
)

config = {"configurable": {"thread_id": "shiva-blog"}}   # 2. which

mem_agent.invoke({"messages": [{"role": "user", "content": "My name is Shiva."}]}, config)
out = mem_agent.invoke({"messages": [{"role": "user", "content": "What is my name?"}]}, config)
print(out["messages"][-1].content)

Your name is Shiva.


`config` is a positional second argument to `invoke`, and `thread_id` lives inside
`configurable`. That nesting is not decorative — `configurable` is the bag of values LangGraph
threads through the whole run.

`InMemorySaver` is a Python dict. Kernel restarts, memory gone. For production swap in
`PostgresSaver` or `SqliteSaver` — same interface, so nothing else in your code changes.

### 4a. Look inside the memory

`get_state` returns exactly what the checkpointer holds for that thread:

In [11]:
snapshot = mem_agent.get_state(config)

print("keys in state :", list(snapshot.values.keys()))
print("messages saved:", len(snapshot.values["messages"]))
for m in snapshot.values["messages"]:
    print(f"  {type(m).__name__:14s} {(m.content or '')[:55]}")
print("\nnext node to run:", snapshot.next or "(none - run finished)")
print("checkpoint id   :", snapshot.config["configurable"]["checkpoint_id"])

keys in state : ['messages']
messages saved: 4
  HumanMessage   My name is Shiva.
  AIMessage      Hello Shiva! How can I assist you today?
  HumanMessage   What is my name?
  AIMessage      Your name is Shiva.

next node to run: (none - run finished)
checkpoint id   : 1f1a850d-01ef-624d-8004-18d913655c0a


Read that output carefully — it tells you what memory *is* in LangChain:

- The state has **one key, `messages`**. Memory is the message list. Nothing else.
- The system prompt is not in there (section 3) — it is re-added on each model call.
- On the second turn the agent did not "recall" anything. It was handed all four earlier
  messages and answered a five-message conversation. **Memory is context, not recall.**
- `next` is empty and there is a `checkpoint_id`: the thread is a resumable position, not just a log.

Every step writes a checkpoint, so a thread has a history you can walk:

In [12]:
history = list(mem_agent.get_state_history(config))
print("checkpoints in this thread:", len(history))
for h in history[:3]:
    print(f"  {len(h.values.get('messages', [])):2d} messages | next={h.next}")

checkpoints in this thread: 6
   4 messages | next=()
   3 messages | next=('model',)
   2 messages | next=('__start__',)


### 4b. Why `thread_id` matters

Same agent object, different `thread_id`:

In [13]:
other = mem_agent.invoke(
    {"messages": [{"role": "user", "content": "What is my name?"}]},
    {"configurable": {"thread_id": "someone-else"}},
)
print(other["messages"][-1].content)

I don't have information about your name. Could you please tell me?


One agent, two sealed conversations. `thread_id` is the whole isolation mechanism, and that makes it
four things at once:

1. **The privacy boundary.** Reuse one `thread_id` across two users and user B reads user A's
   conversation. There is no second check — the id *is* the permission.
2. **The scope of "remember".** New id = clean slate. Same id = continues. Deciding what a
   thread means (per user? per chat window? per support ticket?) is a product decision you make
   by choosing this string.
3. **The resume handle.** After a crash, a redeploy, or a week later, the same id picks the
   conversation back up mid-run — that is why `next` exists in the snapshot above.
4. **The cost dial.** A thread only grows. A month-old `thread_id` sends a month of messages on
   every turn, and you pay for all of them. Long-lived threads need trimming or summarisation.

Rule of thumb: derive it, never invent it — `f"{user_id}:{conversation_id}"`.

---
## 5. Invocation: `invoke`, `batch`, `stream`

Three ways to run the same agent. They differ only in *when* you get output.

### 5a. `invoke` — one input, wait, get everything

In [14]:
r = agent.invoke({"messages": [{"role": "user", "content": "Slugify 'Ten Ideas About Agents'."}]})
print(r["messages"][-1].content)

The slug for 'Ten Ideas About Agents' is "ten-ideas-about-agents".


Default choice. Use it for backend jobs, tests, and anything where a human is not watching a screen.

### 5b. `batch` — many independent inputs, in parallel

In [15]:
import time

titles = ["Vector Databases Explained", "Why Agents Fail", "RAG In Production"]
inputs = [{"messages": [{"role": "user", "content": f"Slugify '{t}'."}]} for t in titles]

t0 = time.time()
serial = [agent.invoke(i) for i in inputs]          # the obvious way
t_serial = time.time() - t0

t0 = time.time()
results = agent.batch(inputs)                       # the same work, concurrently
t_batch = time.time() - t0

print(f"for loop + invoke : {t_serial:.1f}s")
print(f"batch             : {t_batch:.1f}s\n")
for t, r in zip(titles, results):
    print(f"  {t:32s} -> {r['messages'][-1].content}")

for loop + invoke : 5.8s
batch             : 3.0s

  Vector Databases Explained       -> The slug for 'Vector Databases Explained' is "vector-databases-explained".
  Why Agents Fail                  -> The slug for 'Why Agents Fail' is "why-agents-fail".
  RAG In Production                -> The slug for 'RAG In Production' is "rag-in-production".


Same requests, same results — but the `for` loop waits for the **sum** of the calls while `batch`
waits for the **slowest** one. That gap is the entire point, and it widens with every extra input.

`batch` returns results **in input order**, so `zip`ping them back onto your inputs is safe.

Two things to get right:

- The inputs must be **independent**. `batch` is parallel, so item 2 cannot see item 1.
- With memory, pass a **list of configs** — one per input — or every item lands in the same thread
  and they corrupt each other:

  ```python
  mem_agent.batch(inputs, [{"configurable": {"thread_id": t}} for t in ["a", "b", "c"]])
  ```

Use it for bulk work: 500 tickets to classify, 40 titles to slugify.

### 5c. `stream(stream_mode="updates")` — one event per agent step

In [16]:
question = "How many words is the phrase 'agents are just loops with tools', and slugify it too?"

for chunk in agent.stream({"messages": [{"role": "user", "content": question}]},
                          stream_mode="updates"):
    for node, update in chunk.items():
        msg = update["messages"][-1]
        calls = [c["name"] for c in getattr(msg, "tool_calls", [])]
        detail = f"wants {calls}" if calls else (msg.content or "")[:60]
        print(f"[{node:6s}] {type(msg).__name__:14s} {detail}")

[model ] AIMessage      wants ['word_count', 'slugify']
[tools ] ToolMessage    6
[tools ] ToolMessage    agents-are-just-loops-with-tools


[model ] AIMessage      The phrase "agents are just loops with tools" has 6 words, a


This is the loop from section 3, printed as it happens: model asks for tools → tools run → model
answers. Two uses, both real:

- **Debugging.** When an agent gives a wrong answer, this shows you *where* — bad tool choice, bad
  tool arguments, or bad reasoning over a good result.
- **Progress UI.** "Searching…", "Checking word count…" — the strings your users see while they wait.

### 5d. `stream(stream_mode="messages")` — token by token

In [17]:
for token, meta in agent.stream(
    {"messages": [{"role": "user", "content": "In two sentences, what is an agent?"}]},
    stream_mode="messages",
):
    if token.content:
        print(token.content, end="", flush=True)
print()

An

 agent

 is

 an

 entity

...

 decision

-making

 and

 action

.

This is the ChatGPT typewriter effect. It changes nothing about the answer — only when the first
character appears, which is the difference between "fast" and "broken" to a user waiting on a
20-second answer.

Note it yields `(token, metadata)` pairs, and it streams tokens from **every** model call in the
loop — including intermediate ones. Filter on `meta["langgraph_node"]` if you only want the final answer.

### 5e. Choosing

| Method | Output arrives | Use when |
|---|---|---|
| `invoke` | once, at the end | backend jobs, tests, tool-to-tool calls |
| `batch` | once, all of them | many independent inputs — bulk classification, migrations |
| `stream("updates")` | after each step | debugging, progress indicators, audit trails |
| `stream("messages")` | after each token | chat UIs |

All four exist on every agent, and `astream` / `ainvoke` / `abatch` are the async twins. You can also
pass a list: `stream_mode=["updates", "messages"]` when a UI needs both a status line and tokens.

---
## 6. `AgentState`

Section 4 showed that state has one key, `messages`. `AgentState` is the schema of that dict, and
you can add fields to it.

Why would you? Because some things are facts about the run, not turns in a conversation: which
client we are writing for, how many drafts we've produced, a plan the agent is following. Putting
those in the message list means the model has to re-read and re-derive them every single turn.

In [18]:
from langchain.agents import AgentState
from langchain.tools import ToolRuntime
from langgraph.types import Command


class WriterState(AgentState):     # inherits `messages`
    brand_voice: str               # set once, read by tools
    drafts_made: int               # updated by a tool


@tool
def read_brand_voice(runtime: ToolRuntime[None, WriterState]) -> str:
    """Return the brand voice this session must write in."""
    return runtime.state.get("brand_voice", "(not set)")


@tool
def log_draft(runtime: ToolRuntime[None, WriterState]) -> Command:
    """Record that one more draft was finished."""
    n = runtime.state.get("drafts_made", 0) + 1
    return Command(update={
        "drafts_made": n,
        "messages": [{"role": "tool", "content": f"logged, drafts_made={n}",
                      "tool_call_id": runtime.tool_call_id}],
    })


state_agent = create_agent(
    model=llm,
    tools=[read_brand_voice, log_draft],
    system_prompt="Use your tools. Be brief.",
    state_schema=WriterState,          # <- tell the agent about the new fields
)

out = state_agent.invoke({
    "messages": [{"role": "user", "content": "What voice am I writing in? Then log a draft."}],
    "brand_voice": "plain and technical, no hype",
    "drafts_made": 0,
})

print(out["messages"][-1].content)
print("\nstate now:", {k: v for k, v in out.items() if k != "messages"})

You are writing in a plain and technical voice, with no hype. A draft has also been logged.

state now: {'brand_voice': 'plain and technical, no hype', 'drafts_made': 1}


Two directions, and both are in that output:

- **Reading** — add `runtime: ToolRuntime` to a tool signature and use `runtime.state`. LangChain
  injects it; it is *not* part of the schema sent to the model, so the model never sees or invents it.
  `brand_voice` reached the tool without ever being said in the conversation.
- **Writing** — return a `Command(update={...})`. `drafts_made` went `0 -> 1`. When a tool returns a
  `Command`, you must include the `messages` entry with `runtime.tool_call_id` yourself, because you
  have taken over what the tool returns.

Note `messages` is **append-only** while `drafts_made` was **replaced**. That is the built-in
behaviour of the `messages` field and the reason it never needs a merge strategy.

### 6a. `AgentState` vs. memory

This is the distinction people get wrong, so state it plainly: **they are not alternatives.**

`AgentState` is the *shape* of the data. The checkpointer is the *persistence* of it.
Add a `checkpointer` and your custom fields get saved and restored per `thread_id` too — for free.

| | `AgentState` | Memory (checkpointer + `thread_id`) |
|---|---|---|
| What it is | the schema — which keys exist | the storage — writes state after every step |
| Lives for | one run, unless a checkpointer saves it | the life of the thread |
| Set by | you, in the `invoke` dict; tools, via `Command` | LangGraph, automatically |
| Read by | tools and middleware, via `runtime.state` | the agent, when you pass the `thread_id` |
| Without the other | works — you just pass values in every call | works — you just get `messages` |

**What belongs in `AgentState` fields:** structured facts for *this* run that tools need and the
model shouldn't have to parse out of prose — `client_id`, `brand_voice`, `retry_count`, a
`plan` list, `documents_retrieved`, a `budget_remaining` counter.

**What belongs in `messages` (memory):** the conversation. Anything the model needs to *read* as
language — what the user asked, what it answered, what a tool returned.

**What belongs in neither:** facts that must survive across threads — a user's preferences, past
purchases, their name from six months ago. That is long-term memory, and it goes in a `store`
(`InMemoryStore`, `PostgresStore`), which is namespaced by user instead of by thread.

Quick test: *would the model need to re-read this as a sentence every turn?* If yes → messages.
If a tool just needs the value → an `AgentState` field.

---
## 7. Structured output

`result["messages"][-1].content` is a string. If the next line of your code is a database insert or
an API call, a string is a problem — you end up writing regexes against prose.

Declare a Pydantic model instead and hand it to `response_format`.

In [19]:
from pydantic import BaseModel, Field


class Headline(BaseModel):
    """A headline proposal for a blog post."""
    title: str = Field(description="The headline itself.")
    words: int = Field(description="Word count of the headline, from the word_count tool.")
    tone: str = Field(description="One word: the tone of the headline.")


headline_agent = create_agent(
    model=llm,
    tools=[word_count],
    system_prompt="Propose one headline. Use word_count for the count - do not estimate.",
    response_format=Headline,          # <- that is the whole change
)

out = headline_agent.invoke(
    {"messages": [{"role": "user", "content": "A post about vector databases for developers."}]})

article = out["structured_response"]
print(type(article))
print(article)
print("\nfield access:", article.title, "|", article.words, "|", article.tone)

<class '__main__.Headline'>
title='Vector Databases for Developers: Unlocking the Power of High-Dimensional Data' words=10 tone='informative'

field access: Vector Databases for Developers: Unlocking the Power of High-Dimensional Data | 10 | informative


Three things to take from that:

- The result is in a **new key, `structured_response`** — not in `messages`. `messages` still holds
  the full trace, so structured output is added, not substituted.
- You get back a **real `Headline` instance**, already validated by Pydantic. `article.words` is an
  `int`. If the model had returned `"seven"`, you would get a validation error rather than a bug
  three functions downstream.
- `Field(description=...)` is sent to the model. It is prompt text, so write it as an instruction
  ("from the word_count tool"), not as a code comment.

### 7a. The two strategies

`response_format=Headline` picks a strategy for you:

- **`ProviderStrategy`** — the provider constrains its own decoding (OpenAI, Anthropic, Gemini).
  Chosen automatically when supported. Most reliable; the schema cannot be violated.
- **`ToolStrategy`** — LangChain adds one extra tool whose arguments *are* your schema, and the
  model "calls" it to answer. Works with any tool-calling model.

Name the strategy explicitly when you need to override the default — an open-weights model behind a
proxy, say, where the provider advertises native support it doesn't really have:

In [20]:
from langchain.agents.structured_output import ToolStrategy

forced = create_agent(
    model=llm,
    tools=[word_count],
    system_prompt="Propose one headline. Use word_count for the count.",
    response_format=ToolStrategy(Headline),
)
out = forced.invoke({"messages": [{"role": "user", "content": "A post about agent memory."}]})
print(out["structured_response"])
print("\ntool calls made:", [c["name"] for m in out["messages"]
                             for c in getattr(m, "tool_calls", [])])

title='Understanding Agent Memory: Enhancing AI Intelligence and Adaptability' words=69 tone='informative'

tool calls made: ['word_count', 'Headline']


The tool-call list shows the mechanism: `word_count` is yours, `Headline` is the one `ToolStrategy`
injected. That extra call is the cost of the strategy — one more round trip, and one more tool
competing for the model's attention.

And look closely at the `words` value: it is wrong. The headline is not that many words long. That
is not a bug in LangChain — it is the difference between the two strategies. `ToolStrategy` asks the
model to *fill in* the schema as tool arguments, so the model can fill it in badly. `ProviderStrategy`
constrains decoding at the provider, so the shape is guaranteed. Pydantic validated the **type**
here (`words` is an `int`) — nothing can validate that the number is *true*.

Two lessons: let LangChain pick the strategy unless you have a reason not to, and never ask a schema
field for a value a tool already computed — read it from the tool result in your own code instead.

---
## 8. Use case: a content writer agent

Everything above, in one agent that does a real job.

**The brief.** An agency writes blog posts for several clients. Each client has a voice, an
audience, and words they refuse to publish. The agent must write on brief, check itself, log its
work, and hand back a publishable object — and it must survive a "make it shorter" follow-up.

**How the pieces map:**

| Requirement | Piece |
|---|---|
| whose brand are we writing for | `AgentState` field — a fact, not conversation |
| look up voice / banned words | tools reading `runtime.state` |
| count how many drafts we shipped | tool returning `Command(update=...)` |
| the workflow to follow | `system_prompt` |
| handle "now make it shorter" | checkpointer + `thread_id` |
| output the CMS can accept | `response_format` with Pydantic |

In [21]:
BRAND_GUIDES = {
    "nunnari": {
        "voice": "plain and technical, short sentences, no hype",
        "audience": "working developers who already know Python",
        "banned": ["revolutionary", "game-changing", "unleash", "seamless"],
    },
    "brightpath": {
        "voice": "warm and encouraging, second person",
        "audience": "non-technical founders",
        "banned": ["leverage", "synergy", "utilize"],
    },
}


class ContentState(AgentState):
    client: str          # which brand guide applies to this thread
    drafts_made: int     # how many finished drafts this thread produced


@tool
def get_brand_guide(runtime: ToolRuntime[None, ContentState]) -> dict:
    """Get the voice, audience and banned words for the client of this session."""
    return BRAND_GUIDES[runtime.state["client"]]


@tool
def check_banned_words(draft: str, runtime: ToolRuntime[None, ContentState]) -> str:
    """Check a draft against the client's banned word list. Call this before finishing."""
    banned = BRAND_GUIDES[runtime.state["client"]]["banned"]
    hits = [w for w in banned if w in draft.lower()]
    return "clean - no banned words" if not hits else f"REWRITE, banned words present: {hits}"


@tool
def log_draft(title: str, runtime: ToolRuntime[None, ContentState]) -> Command:
    """Record a finished draft. Call this exactly once, after the banned-word check passes."""
    n = runtime.state.get("drafts_made", 0) + 1
    return Command(update={
        "drafts_made": n,
        "messages": [{"role": "tool", "content": f"draft #{n} logged: {title}",
                      "tool_call_id": runtime.tool_call_id}],
    })


class BlogPost(BaseModel):
    """A blog post ready to be posted to the CMS."""
    title: str = Field(description="Headline, under 12 words.")
    slug: str = Field(description="URL slug, lowercase and hyphenated.")
    meta_description: str = Field(description="SEO description, 150 characters or less.")
    body: str = Field(description="The post body in markdown.")
    word_count: int = Field(description="Body word count, from the word_count tool.")
    tags: list[str] = Field(description="Two to four lowercase topic tags.")


print("tools:", [t.name for t in [get_brand_guide, check_banned_words, log_draft, word_count, slug_tool]])

tools: ['get_brand_guide', 'check_banned_words', 'log_draft', 'word_count', 'slugify']


In [22]:
CONTENT_SYSTEM = """You are a staff writer at a content agency.

Follow this order every time, without being asked:
1. Call get_brand_guide to learn the client's voice, audience and banned words.
2. Write the post in that voice, for that audience.
3. Call check_banned_words on your draft. If it says REWRITE, rewrite and check again.
4. Call word_count on the body and slugify on the title.
5. Call log_draft once.
Then return the BlogPost.

Keep posts to roughly the length asked for. Never invent statistics."""

writer = create_agent(
    model=llm,
    tools=[get_brand_guide, check_banned_words, log_draft, word_count, slug_tool],
    system_prompt=CONTENT_SYSTEM,
    state_schema=ContentState,
    response_format=BlogPost,
    checkpointer=InMemorySaver(),
)

thread = {"configurable": {"thread_id": "nunnari-2026-09"}}

out = writer.invoke({
    "messages": [{"role": "user",
                  "content": "Write a ~120 word post on why agents need a thread_id."}],
    "client": "nunnari",
    "drafts_made": 0,
}, thread)

post = out["structured_response"]
print(f"TITLE  : {post.title}")
print(f"SLUG   : {post.slug}")
print(f"META   : {post.meta_description}")
print(f"WORDS  : {post.word_count}")
print(f"TAGS   : {post.tags}")
print(f"\n{post.body}")

TITLE  : Why Agents Need a Thread_id
SLUG   : why-agents-need-a-thread_id
META   : Learn why a thread_id is essential for managing conversations and maintaining context in agent systems.
WORDS  : 120
TAGS   : ['agents', 'thread_id', 'conversations', 'context']

Agents need a thread_id to manage conversations effectively. A thread_id uniquely identifies each conversation thread, allowing the agent to track context and maintain continuity. Without a thread_id, it becomes difficult to associate messages with the correct conversation, especially when handling multiple interactions simultaneously. This can lead to confusion and errors in processing user inputs. Using a thread_id also helps in logging and debugging by providing a clear reference to specific conversation flows. In multi-agent systems, thread_ids ensure that messages are routed correctly between agents. Overall, a thread_id is essential for organizing and managing conversations in a structured and reliable way.


Now the follow-up — the one that would be impossible without memory. Note what is *not* in the
next call: no client, no brief, no draft. Just an instruction that only makes sense in context.

> You will see a warning about *"Deserializing unregistered type `__main__.BlogPost`"*. It is
> harmless here and it is telling you something true: the checkpointer had to save your Pydantic
> object, and classes defined in a notebook cell aren't importable by name. In a real project
> `BlogPost` lives in a module, and the warning disappears.

In [23]:
out2 = writer.invoke(
    {"messages": [{"role": "user", "content": "Too long. Cut it to about 60 words and make the title punchier."}]},
    thread,
)
post2 = out2["structured_response"]
print(f"TITLE : {post2.title}")
print(f"WORDS : {post2.word_count}  (was {post.word_count})")
print(f"\n{post2.body}")
print("\nstate:", {k: v for k, v in out2.items() if k not in ("messages", "structured_response")})

Deserializing unregistered type __main__.BlogPost from checkpoint. This will be blocked in a future version. Set LANGGRAPH_STRICT_MSGPACK=true to block now, or add to allowed_msgpack_modules to allow explicitly: [('__main__', 'BlogPost')]


TITLE : Why Thread_id Matters for Agents
WORDS : 49  (was 120)

Thread_id is crucial for agents to track and manage conversations. It uniquely identifies each thread, ensuring messages link to the right context. This prevents confusion and errors, especially with multiple interactions. Thread_ids also aid in logging and routing messages correctly in multi-agent systems. They keep conversations organized and reliable.

state: {'client': 'nunnari', 'drafts_made': 2}


`drafts_made` is at 2 — the state field survived the second call because the checkpointer saved it
alongside the messages. That is section 6a in one number: `AgentState` gave the field a shape, and
memory gave it a lifetime.

Same agent object, different client, different thread:

In [24]:
out3 = writer.invoke({
    "messages": [{"role": "user", "content": "Write a ~70 word post on why AI projects stall."}],
    "client": "brightpath",
    "drafts_made": 0,
}, {"configurable": {"thread_id": "brightpath-2026-09"}})

p3 = out3["structured_response"]
print(f"TITLE : {p3.title}\n")
print(p3.body)
print("\ndrafts_made on this thread:", out3["drafts_made"])

TITLE : Why AI Projects Stall

AI projects often stall because they can feel overwhelming, especially if you're new to the technology. Common hurdles include unclear goals, lack of data, or insufficient expertise. It's important to start small, focus on a clear problem, and build your team's skills gradually. Remember, patience and persistence are key. With the right approach, you can overcome these challenges and bring your AI vision to life.

drafts_made on this thread: 1


Different voice, and `drafts_made` back to 1 — a different `thread_id` means a different world.
One deployed agent, many clients, no cross-contamination.

Look at the trace once to see the workflow actually executing in order:

In [25]:
for m in out3["messages"]:
    calls = [c["name"] for c in getattr(m, "tool_calls", [])]
    label = f"-> {calls}" if calls else (m.content or "")[:70].replace("\n", " ")
    print(f"{type(m).__name__:14s} {label}")

HumanMessage   Write a ~70 word post on why AI projects stall.
AIMessage      -> ['get_brand_guide']
ToolMessage    {"voice": "warm and encouraging, second person", "audience": "non-tech
AIMessage      -> ['check_banned_words']
ToolMessage    clean - no banned words
AIMessage      -> ['word_count', 'slugify']
ToolMessage    65
ToolMessage    why-ai-projects-stall
AIMessage      -> ['log_draft']
ToolMessage    draft #1 logged: Why AI Projects Stall
AIMessage      {"title":"Why AI Projects Stall","slug":"why-ai-projects-stall","meta_


---
## Recap

```python
llm    = init_chat_model("openai:gpt-4.1-mini")      # 1. model, provider in a string
@tool                                                # 2. tools, docstring = when to use
def my_tool(x: str) -> str: ...
agent  = create_agent(
    model=llm,
    tools=[my_tool],
    system_prompt="...",         # 3. the job, re-sent every model call
    state_schema=MyState,        # 6. extra fields tools can read and write
    response_format=MySchema,    # 7. -> result["structured_response"]
    checkpointer=InMemorySaver() # 4. memory, needs a thread_id
)
agent.invoke(inp, {"configurable": {"thread_id": "user-42"}})   # 5. or batch / stream
```

The six things worth remembering:

1. `system_prompt` is not a message — it is re-sent on every model call.
2. An agent is stateless until you give it a checkpointer, and useless until you give it a `thread_id`.
3. Memory is context, not recall: the model re-reads the whole thread every turn. Threads cost money.
4. `thread_id` is your privacy boundary. Derive it from real ids.
5. `AgentState` is shape, the checkpointer is lifetime. Structured facts go in fields; language goes in messages.
6. `response_format` gives you a validated object, in `structured_response`.

### Exercises

1. Give `check_banned_words` a bad description ("checks stuff") and rerun. Watch the agent skip it.
   That is how much the docstring is doing.
2. Add a `max_words: int` field to `ContentState` and have the writer enforce it in a tool rather
   than in the prompt. Which one does the model obey more reliably?
3. Run the writer with `stream_mode="updates"` and print each step. Where does it waste a turn?
4. Reuse `thread_id="nunnari-2026-09"` but pass `client="brightpath"`. Predict the result before
   running it, then explain what you see.

Next notebook: turning agents into tools for other agents — the subagent pattern.